In [ ]:
"""
03_hyperparameter_tuning.py -- ScoutAI Model Optimization (Dual Model)

Runs a RandomizedSearchCV over the XGBoost regressor for BOTH the 
'full' and 'performance_only' models to find their optimal hyperparameters.
Compares the tuned models against the existing production models and 
saves the new versions only if they achieve a lower RMSE.
Outputs tuning results to the data directory.
"""

import os
import sys
import pandas as pd
import numpy as np
import sqlalchemy
import joblib
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
import time
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

MODELS_DIR = "models"
DATA_DIR = "data"

# Ensure output directories exist
for directory in [MODELS_DIR, DATA_DIR]:
    os.makedirs(directory, exist_ok=True)

def main():
    # ==========================================
    # 1. DATABASE CONNECTION & DATA LOAD
    # ==========================================
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Fetching data for Hyperparameter Tuning...")
    try:
        engine = sqlalchemy.create_engine(db_url)
        df = pd.read_sql("SELECT * FROM view_scout_master", engine)
    except Exception as e:
        print(f"[ERROR] Database connection failed: {e}")
        sys.exit(1)

    df['current_market_value'] = pd.to_numeric(df['current_market_value'], errors='coerce').fillna(0)
    df = df[df['current_market_value'] > 0].copy()

    # Target is log-transformed to handle extreme outliers properly
    y = np.log1p(df['current_market_value'])

    # ==========================================
    # 2. FEATURE ENGINEERING
    # ==========================================
    print("[SYSTEM] Engineering features...")
    cols_to_clean = [
        'age', 'total_appearances', 'international_caps', 'international_goals',
        'max_career_transfer_fee', 'most_recent_transfer_fee', 'height_in_cm',
        'minutes_per_match', 'total_goals', 'total_assists', 'goals_per_90',
        'assists_per_90', 'total_yellow_cards', 'total_red_cards', 'club_squad_size',
        'club_avg_age', 'stadium_seats', 'foreigners_percentage', 'contract_months_remaining',
    ]
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)

    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15),
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60),
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')

    league_weights = {
        'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3,
        'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05,
    }
    df['league_coefficient'] = df['competition_name'].map(league_weights).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)

    FULL_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
        'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
        'detailed_position', 'foot', 'passport_tier',
    ]

    PERFORMANCE_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier',
    ]

    models_to_tune = [
        {"label": "full", "features": FULL_FEATURES},
        {"label": "performance_only", "features": PERFORMANCE_FEATURES}
    ]

    report_lines = []
    header = (
        "==========================================\n"
        " 🛠️ SCOUT AI: DUAL-MODEL OPTIMIZATION 🛠️\n"
        "==========================================\n"
    )
    print(f"\n{header}", end="")
    report_lines.append(header)

    # ==========================================
    # 3 & 4. HYPERPARAMETER TUNING LOOP
    # ==========================================
    for model_info in models_to_tune:
        model_label = model_info["label"]
        features = model_info["features"]
        
        print(f"\n[SYSTEM] Starting optimization for '{model_label}' model...")

        X = df[features]
        # Match the random_state exactly with the training script to ensure identical holdout set
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        model_path = os.path.join(MODELS_DIR, f'scout_model_{model_label}.pkl')
        try:
            old_pipeline = joblib.load(model_path)
        except FileNotFoundError:
            print(f"[ERROR] Could not find '{model_path}'. Skipping {model_label} tuning.")
            continue
            
        preprocessor = old_pipeline.named_steps['preprocessor']
        regressor = old_pipeline.named_steps['regressor']

        old_preds = old_pipeline.predict(X_test)
        old_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(old_preds)))

        print(f"[SYSTEM] Running 5-Fold Cross-Validation Tuning for {model_label} (this may take a minute)...")
        start_time = time.time()

        param_distributions = {
            'regressor__n_estimators': [400, 800, 1200, 1600],
            'regressor__max_depth': [4, 5, 6, 7, 8],
            'regressor__learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08],
            'regressor__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
            'regressor__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
            'regressor__min_child_weight': [1, 3, 5, 7],
            'regressor__gamma': [0, 0.1, 0.3, 0.5],
            'regressor__reg_alpha': [0, 0.1, 1],
            'regressor__reg_lambda': [1, 1.5, 2],
        }

        tuning_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', regressor)
        ])

        # n_jobs=1 to avoid the Windows crash seen with n_jobs=-1
        random_search = RandomizedSearchCV(
            estimator=tuning_pipeline,
            param_distributions=param_distributions,
            n_iter=25,
            cv=5,
            scoring='neg_mean_squared_error',
            random_state=42,
            n_jobs=1,
            verbose=1,
        )

        random_search.fit(X_train, y_train)

        elapsed_time = time.time() - start_time
        print(f"[SYSTEM] {model_label.title()} tuning completed in {elapsed_time:.1f} seconds!")

        # ==========================================
        # 5. EVALUATE & SAVE THE NEW TUNED MODEL
        # ==========================================
        best_pipeline = random_search.best_estimator_
        new_preds = best_pipeline.predict(X_test)
        new_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(new_preds)))

        section_header = f"\n--- RESULTS: {model_label.upper()} MODEL ---\nBest Parameters Found:\n"
        print(section_header, end="")
        report_lines.append(section_header)
        
        for key, value in random_search.best_params_.items():
            clean_key = key.replace('regressor__', '')
            param_str = f"  * {clean_key}: {value}\n"
            print(param_str, end="")
            report_lines.append(param_str)
            
        metrics_str = (
            "------------------------------------------\n"
            f" Old Model RMSE: €{old_rmse:,.0f}\n"
            f" New Tuned RMSE: €{new_rmse:,.0f}\n"
        )
        print(metrics_str, end="")
        report_lines.append(metrics_str)

        if new_rmse < old_rmse:
            success_msg = f"✅ SUCCESS: The tuned model is better! Saving as 'scout_model_{model_label}_tuned.pkl'...\n"
            print(success_msg, end="")
            report_lines.append(success_msg)
            
            new_model_path = os.path.join(MODELS_DIR, f'scout_model_{model_label}_tuned.pkl')
            joblib.dump(best_pipeline, new_model_path)
        else:
            fail_msg = f"⚠️ NOTE: The old {model_label} model performed at least as well. Keeping the old model.\n"
            print(fail_msg, end="")
            report_lines.append(fail_msg)

    # Save report to TXT in DATA_DIR
    txt_export_path = os.path.join(DATA_DIR, 'tuning_results_log.txt')
    with open(txt_export_path, 'w', encoding='utf-8') as f:
        f.writelines(report_lines)
    
    print(f"\n[SYSTEM] Tuning results securely saved to '{txt_export_path}'")

if __name__ == "__main__":
    main()

[SYSTEM] Fetching data for Hyperparameter Tuning...
